<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

One row = one content item (content_hash_id), on one calendar day (report_date), scoped to one
client (client_hash_id) — the grain of fact_content_daily_performance.

Table(s): dim_content (content metadata, joined on content_hash_id), fact_content_daily_performance
(the daily fact table, partitioned by month), and dim_clients (for gsc_data_start / ga4_data_start,
needed to know each client's real history window).

Time window: iterating on month=2026-03 (a mid-panel month), per the warehouse warning — the
_sample table is June 2026, the panel's final month, which is the natural outcome window of any
past->future label, so it's sealed for testing only, never for developing label logic.

In [6]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:14} {n:>12,} rows')

Paste your Hugging Face READ token (hf_...): ··········
dim_clients             104 rows
dim_content         519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily       78,835,655 rows
fact_query_90d    2,414,248 rows


Feature (knowable before the decision moment): prior-window impressions/clicks/position from
fact_content_daily_performance, query-mix signals from fact_content_query_90d (visible_queries,
rare_share, anon_share, top_query_share) -- all computed from data strictly before the label
window.

Label/proxy: is_declining = did impressions drop >=20% from the prior 30 days to the last 30
days within the window I'm studying. Computed FROM the outcome window -- never a feature.

Context (grouping/joining only, never modeled on): content_hash_id, client_hash_id,
report_date -- IDs, never signal.

Excluded (with why): ga4_* columns for rows before a client's ga4_data_start -- these are
zero-filled placeholders, not real zero engagement, so treating them as a real value would
inject fake signal. Excluded via the ga4_data_available flag until proven safe to include.

In [3]:
# Confirm the real column names backing the classification above, instead of guessing them.
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 1").df()[['column_name','column_type']])
print()
print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()[['column_name','column_type']])

dim_content columns:
                   column_name column_type
0               client_hash_id     VARCHAR
1              content_hash_id     VARCHAR
2              keyword_hash_id     VARCHAR
3                  url_hash_id     VARCHAR
4           keyword_char_count      BIGINT
5          keyword_token_count      BIGINT
6               url_char_count      BIGINT
7         content_created_date        DATE
8         content_updated_date        DATE
9                 content_type     VARCHAR
10               search_volume      BIGINT
11                 competition      DOUBLE
12           competition_level     VARCHAR
13                         cpc      DOUBLE
14                 main_intent     VARCHAR
15                   backlinks      BIGINT
16              category_count      BIGINT
17        keyword_created_date        DATE
18               provider_used     VARCHAR
19                  model_used     VARCHAR
20                  char_count      BIGINT
21                  word_count   

In [7]:
MONTH = "2026-03"

# ---- Query A: grain check ----
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Query A -- grain violations found: {len(grain_check)} (0 means the grain holds)")

# ---- Query B: row count + date span for this month's slice ----
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
""").df()
print("Query B -- row count and date span:")
print(span)

# ---- Query C: availability, filtered with IS TRUE ----
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
""").df()
print("Query C -- availability:")
print(avail)
print(f"Share surviving IS TRUE filter: {avail['rows_with_ga4'][0] / avail['total_rows'][0]:.1%}")

# ---- Five features, built on the same month ----
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_month,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impressions_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH}-01' AND report_date < DATE '{MONTH}-01' + INTERVAL 1 MONTH
    GROUP BY 1,2
    HAVING impressions_month >= 100
""").df()
print(f"\nFeature frame: {len(features):,} content items")
features.head()

# Available-when, one line per feature:
# impressions_month    -- knowable at month-end, from that month's own logged data
# clicks_month          -- same: logged continuously through the month
# avg_position_month    -- same: GSC reports this daily, no future data needed
# ctr_month              -- derived only from the two columns above, same window
# impressions_with_ga4  -- knowable the moment ga4_data_available is set, no lookahead

# ---- The trap: add a label-derived column on purpose ----
import numpy as np
features['is_declining'] = (features['ctr_month'] < features['ctr_month'].median()).astype(int)

# Deliberately leaked feature: derived directly from the label itself
features['LEAKY_ctr_rank'] = features['ctr_month'].rank(pct=True)  # same signal the label is built from

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_leaky = features[['impressions_month', 'avg_position_month', 'LEAKY_ctr_rank']].fillna(0)
y = features['is_declining']
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.25, random_state=42)
leaky_score = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr).score(Xte, yte)
print(f"\nWITH leak (LEAKY_ctr_rank included): accuracy = {leaky_score:.3f}")

# Remove it, keep the honest number
X_honest = features[['impressions_month', 'avg_position_month']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.25, random_state=42)
honest_score = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr).score(Xte, yte)
print(f"WITHOUT leak (honest features only): accuracy = {honest_score:.3f}")
print(f"\nLeakage inflated accuracy by {leaky_score - honest_score:.3f} -- LEAKY_ctr_rank is just")
print("ctr_month's own rank, so it's trivially predictive of a label built from ctr_month. Deleted.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query A -- grain violations found: 0 (0 means the grain holds)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query B -- row count and date span:
    n_rows      min_d      max_d  n_clients  n_content
0  9841378 2026-03-01 2026-03-31         55     331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query C -- availability:
   total_rows  rows_with_ga4
0     9841378       413966.0
Share surviving IS TRUE filter: 4.2%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Feature frame: 101,441 content items

WITH leak (LEAKY_ctr_rank included): accuracy = 1.000
WITHOUT leak (honest features only): accuracy = 0.630

Leakage inflated accuracy by 0.370 -- LEAKY_ctr_rank is just
ctr_month's own rank, so it's trivially predictive of a label built from ctr_month. Deleted.


Named limitation: history depth is a wildly unbalanced panel across clients (gsc_data_start
varies per client in dim_clients), so a fixed calendar month like 2026-03 represents a full
month of real history for some clients and near-zero history for others just onboarded. This
data can never tell me whether a decline is genuinely about the content, or just an artifact of
that client having thin history in this window -- it needs a per-client window check, not a
single global calendar cut, before I'd trust a cross-client comparison.

In [8]:
# Confirms the limitation above with a real number rather than leaving it as an assertion.
history = con.sql(f"""
    SELECT client_hash_id, gsc_data_start
    FROM {TABLES['dim_clients']}
""").df()
import pandas as pd
history['gsc_data_start'] = pd.to_datetime(history['gsc_data_start'])
cutoff = pd.Timestamp('2026-03-01') - pd.Timedelta(days=365)
thin_history_clients = (history['gsc_data_start'] > cutoff).sum()
print(f"{thin_history_clients} of {len(history)} clients have under 12 months of GSC history")
print("before 2026-03 -- confirms the panel is unbalanced, not a uniform history depth.")

64 of 104 clients have under 12 months of GSC history
before 2026-03 -- confirms the panel is unbalanced, not a uniform history depth.
